# LSTM & GRU

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/rnns/03-lstm-and-gru

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — gates build a gradient highway

The vanilla RNN's fatal flaw is that gradients must pass through a *multiplication by `W_hh`* at every
step. The **LSTM** re-architects the memory: it adds a **cell state** `c` updated **additively** —
`c_t = f ⊙ c_{t−1} + i ⊙ g` — where the **forget gate** `f`, **input gate** `i`, and **output gate**
`o` are learned sigmoids that decide what to keep, write, and reveal. Because the cell state's
recurrence is elementwise-multiply-by-`f` (which the network can hold near 1) plus an *addition*,
gradients flow back along it with factor `f` per step instead of `‖W_hh‖·tanh'` — a **gradient
highway**. The **GRU** achieves the same with two gates and no separate cell. We build both from
scratch, verify by hand, and validate against `jax`.

## An LSTM cell, fully from scratch

Three gates and an **additive** cell-state update — the gradient highway that fixes the vanishing gradient from the previous lesson.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

class LSTMCell:
    def __init__(self, n_in, nh):
        k = n_in + nh
        self.nh = nh
        self.Wf=np.random.randn(nh,k)*0.1; self.bf=np.ones((nh,1))   # forget bias=1
        self.Wi=np.random.randn(nh,k)*0.1; self.bi=np.zeros((nh,1))
        self.Wc=np.random.randn(nh,k)*0.1; self.bc=np.zeros((nh,1))
        self.Wo=np.random.randn(nh,k)*0.1; self.bo=np.zeros((nh,1))

    def step(self, x, h, c):
        z = np.vstack([h, x])
        f = sigmoid(self.Wf@z + self.bf)
        i = sigmoid(self.Wi@z + self.bi)
        g = np.tanh(self.Wc@z + self.bc)
        o = sigmoid(self.Wo@z + self.bo)
        c = f*c + i*g                 # additive update = gradient highway
        h = o*np.tanh(c)
        return h, c, dict(f=f, i=i, o=o)

cell = LSTMCell(n_in=3, nh=5)
h = c = np.zeros((5,1))
for t in range(4):
    h, c, gates = cell.step(np.random.randn(3,1), h, c)
    print(f't={t}  mean forget gate={gates["f"].mean():.2f}  ||cell||={np.linalg.norm(c):.2f}')

**What to notice:** each step computes three sigmoid **gates** (forget/input/output) plus a tanh
candidate, then the crucial line: `c = f*c + i*g` — the old memory scaled by the forget gate, **plus**
new content. Note the forget bias initialized to **1**: the gate starts open (`f ≈ 0.73+`), so memory
is retained by default and the network learns what to forget.

## A full LSTM timestep, by hand (pure-Python verification)

The lesson works one scalar LSTM step ($D=1$, $H=1$) entirely by hand. Here we reproduce it with only the standard library (`math`), so the arithmetic is deterministic and checkable without any dependencies.

Inputs: $h_{t-1}=0$, $x_t=1$, $c_{t-1}=2$. Weights are chosen to keep the pre-activations simple.

In [ ]:
import math  # stdlib only -- deterministic, no numpy needed

def sig(z):
    return 1 / (1 + math.exp(-z))

# Inputs
h_prev, x, c_prev = 0.0, 1.0, 2.0

# Weights per gate: (W_h, W_x, bias)
Wf = (0.0, 0.0,  2.0)   # forget  (bias +2 -> defaults to remembering)
Wi = (0.0, 1.0, -1.0)   # input
Wc = (0.0, 1.0,  0.0)   # candidate
Wo = (0.0, 1.0,  0.0)   # output

def preact(W):
    Wh, Wx, b = W
    return Wh * h_prev + Wx * x + b

f  = sig(preact(Wf))                 # forget gate
i  = sig(preact(Wi))                 # input gate
ct = math.tanh(preact(Wc))           # candidate
o  = sig(preact(Wo))                 # output gate

c = f * c_prev + i * ct              # additive cell update
h = o * math.tanh(c)                 # hidden state

print(f"f_t      = {f:.3f}   (expected 0.881)")
print(f"i_t      = {i:.3f}   (expected 0.500)")
print(f"c_tilde  = {ct:.3f}   (expected 0.762)")
print(f"o_t      = {o:.3f}   (expected 0.731)")
print(f"c_t      = {c:.3f}   (expected 2.142)")
print(f"h_t      = {h:.3f}   (expected 0.711)")

# Verification against the hand-computed values in the lesson
assert abs(f  - 0.881) < 1e-3
assert abs(i  - 0.500) < 1e-3
assert abs(ct - 0.762) < 1e-3
assert abs(o  - 0.731) < 1e-3
assert abs(c  - 2.142) < 1e-3
assert abs(h  - 0.711) < 1e-3
print("\nVERIFIED: by-hand LSTM timestep matches the lesson.")

**What to notice:** the pure-Python trace matches the hand calculation gate by gate — every LSTM
number is just sigmoids and tanhs of affine combinations. Follow `c` through the step: it changed by an
elementwise scale-and-add, never a full matrix multiply. That's the architectural difference from the
vanilla RNN, in one line.

## The library way — validate the LSTM step against `jax`

Same discipline as before: reimplement the step in `jax.numpy` and assert it reproduces our NumPy
cell's `h` and `c` exactly (the computation inside `torch.nn.LSTM`).

In [ ]:
import jax.numpy as jnp

def lstm_step_jax(cell, x, h, c):
    z = jnp.vstack([jnp.asarray(h), jnp.asarray(x)])
    sig = lambda v: 1 / (1 + jnp.exp(-v))
    f = sig(jnp.asarray(cell.Wf) @ z + jnp.asarray(cell.bf))
    i = sig(jnp.asarray(cell.Wi) @ z + jnp.asarray(cell.bi))
    g = jnp.tanh(jnp.asarray(cell.Wc) @ z + jnp.asarray(cell.bc))
    o = sig(jnp.asarray(cell.Wo) @ z + jnp.asarray(cell.bo))
    c_new = f * jnp.asarray(c) + i * g
    return o * jnp.tanh(c_new), c_new

np.random.seed(1)
cell_v = LSTMCell(n_in=3, nh=5)
x0 = np.random.randn(3, 1); h0 = np.zeros((5, 1)); c0 = np.zeros((5, 1))
h_np, c_np, _ = cell_v.step(x0, h0, c0)
h_jx, c_jx = lstm_step_jax(cell_v, x0, h0, c0)
assert np.allclose(h_np, np.array(h_jx), atol=1e-6) and np.allclose(c_np, np.array(c_jx), atol=1e-6)
print('our LSTM step == jax reimplementation (== torch.nn.LSTM cell math) ✓')

**What to notice:** bit-for-bit agreement — the LSTM's "complexity" is just four gated affine maps,
identical in any framework. What `torch.nn.LSTM` adds is fused kernels and BPTT bookkeeping, not
different math.

## Parameter count: LSTM vs GRU

Each gate is a linear layer over the concatenation $[\mathbf{h}_{t-1}, \mathbf{x}_t]$, costing $H(D+H+1)$ parameters ($W_x$: $H\times D$, $W_h$: $H\times H$, bias: $H$). An LSTM has **4** such layers (forget, input, candidate, output); a GRU has **3** (reset, update, candidate). Pure-Python count below.

In [ ]:
def gate_params(D, H):
    """Parameters in one gate layer: W_x (H*D) + W_h (H*H) + bias (H)."""
    return H * D + H * H + H

def lstm_params(D, H):
    return 4 * gate_params(D, H)

def gru_params(D, H):
    return 3 * gate_params(D, H)

D, H = 10, 20
per_gate = gate_params(D, H)
lstm_p = lstm_params(D, H)
gru_p = gru_params(D, H)

print(f"D={D}, H={H}")
print(f"per-gate layer : {per_gate}        (= {H}*{D} + {H}*{H} + {H})")
print(f"LSTM (4 gates) : {lstm_p}")
print(f"GRU  (3 gates) : {gru_p}")
print(f"GRU / LSTM     : {gru_p / lstm_p:.2f}   ({(1 - gru_p/lstm_p)*100:.0f}% fewer)")

assert per_gate == 620
assert lstm_p == 2480
assert gru_p == 1860
assert abs(gru_p / lstm_p - 0.75) < 1e-9
print("\nVERIFIED: parameter counts match the lesson (2480 vs 1860, exactly 25% fewer).")

**What to notice:** an LSTM carries **4 gate blocks** (~`4·H·(D+H)` weights) and a GRU **3** — the
GRU is ~25% cheaper for the same hidden size, with usually comparable accuracy. That's the practical
trade: LSTM when you want maximum gating flexibility, GRU when parameters/latency matter.

## The gradient highway, quantified (pure-Python)

Differentiating the additive update gives $\partial c_t / \partial c_{t-1} \approx f_t$, so the gradient over $T$ steps is the **product of forget gates**, $\prod_t f_t \approx f^T$. A vanilla RNN instead scales like (per-step factor)$^T$. The numbers below match the contrast in the lesson.

In [ ]:
T = 50

# LSTM cell-state gradient over T steps = product of forget gates (held constant here)
lstm_open   = 0.95 ** T   # forget gate kept open at 0.95
lstm_wide   = 0.99 ** T   # forget gate kept at 0.99
# Vanilla RNN: gradient scales like (effective recurrent factor)^T
rnn_decay   = 0.90 ** T

print(f"After T={T} steps:")
print(f"  LSTM, forget gate f=0.95 : {lstm_open:.4f}")
print(f"  LSTM, forget gate f=0.99 : {lstm_wide:.4f}")
print(f"  vanilla RNN, factor 0.90 : {rnn_decay:.6f}")
print(f"  LSTM(0.95) is ~{lstm_open / rnn_decay:.0f}x larger than the RNN gradient")

assert abs(lstm_open - 0.0769) < 1e-3
assert abs(lstm_wide - 0.6050) < 1e-3
assert abs(rnn_decay - 0.005154) < 1e-5
assert lstm_open / rnn_decay > 10
print("\nVERIFIED: gradient-highway numbers match the lesson (15x advantage at f=0.95).")

**What to notice:** the arithmetic of the highway — after 50 steps, an LSTM holding its forget gate
at 0.95 retains ~15× more gradient than a vanilla RNN with factor 0.9, and at `f = 0.99` it keeps
**0.6** of the signal where the RNN keeps **0.005**. Because `f` is *learned per-unit per-step*, the
network can hold gates open exactly where long memory is needed.

## A GRU cell, from scratch

Two gates (reset, update), one state vector — the update gate does the forget+input job.

In [ ]:
class GRUCell:
    def __init__(self, n_in, nh):
        k = n_in + nh; self.nh = nh
        self.Wz=np.random.randn(nh,k)*0.1; self.Wr=np.random.randn(nh,k)*0.1
        self.Wh=np.random.randn(nh,k)*0.1

    def step(self, x, h):
        z = sigmoid(self.Wz@np.vstack([h,x]))
        r = sigmoid(self.Wr@np.vstack([h,x]))
        hh = np.tanh(self.Wh@np.vstack([r*h, x]))
        return (1-z)*h + z*hh

gru = GRUCell(3, 5); h = np.zeros((5,1))
for t in range(4):
    h = gru.step(np.random.randn(3,1), h)
print('GRU final hidden:', np.round(h.ravel(), 3))

**What to notice:** the **GRU** merges the forget/input pair into one **update gate** (`z`) and drops
the separate cell state — `h = (1−z)⊙h + z⊙h̃` is the same keep-plus-write pattern with fewer moving
parts. Same highway idea, two gates instead of three.

## The whole point: memory survives long gaps

Compare how a signal injected at $t=0$ persists in an LSTM cell state vs a vanilla RNN hidden state, when the forget gate stays open (≈1). The LSTM keeps it; the RNN washes it out through repeated `tanh` squashing.

In [ ]:
T = 60
# LSTM with forget gate held open and no new input: c_t = c_0
c = np.ones((1,1)); lstm_mem = []
for t in range(T):
    f = sigmoid(np.array([[3.0]]))     # forget gate ~0.95 (bias open)
    c = f*c + 0.0                       # no new input written
    lstm_mem.append(c.item())
# Vanilla RNN with small recurrent weight, signal decays
h = 1.0; rnn_mem = []
for t in range(T):
    h = np.tanh(0.9*h)
    rnn_mem.append(h)
plt.plot(lstm_mem, label='LSTM cell state (gate open)', color='#14b8a6')
plt.plot(rnn_mem, label='vanilla RNN hidden state', color='#f43f5e')
plt.xlabel('time step'); plt.ylabel('retained signal'); plt.legend()
plt.title('Long-term memory: LSTM cell vs vanilla RNN'); plt.show()

**What to notice:** the punchline plot — with its gate held open, the LSTM cell state carries the
signal across all 60 steps essentially undimmed, while the vanilla RNN's hidden state decays toward
its fixed point within ~20. This is the *entire reason* gated RNNs exist: memory that survives long
gaps.

## Gotchas & tradeoffs

- **Gates ≈ 3–4× the compute and parameters** of a vanilla RNN per step — the price of the highway.
- **Initialize the forget bias to 1** (or higher): starting with gates half-closed makes early training
  forget everything; open-by-default is the standard trick (already used above).
- **Still sequential.** LSTMs fix vanishing gradients but not the step-by-step dependency — they can't
  parallelize across time, which is why transformers ultimately displaced them.
- **LSTM vs GRU is empirical.** Neither dominates; GRU is smaller/faster, LSTM slightly more expressive.
  Try both.

In [ ]:
# Forget-bias initialization matters: closed gates (bias 0) forget fast, open gates (bias 1) retain
for bias, label in [(0.0, 'bias 0 (half-closed, f~0.5)'), (1.0, 'bias 1 (open, f~0.73)'), (2.0, 'bias 2 (f~0.88)')]:
    f = 1 / (1 + np.exp(-bias))
    print(f'{label:30}: signal kept after 20 steps = {f**20:.4f}')
print('\n-> with bias 0 the memory halves every step; open-by-default (bias>=1) preserves early training signal')

**What to notice:** a forget gate at its bias-0 default (`f ≈ 0.5`) wipes out all but `1e-6` of the
signal in 20 steps, while bias 1 keeps ~0.2% and bias 2 keeps ~8% — orders of magnitude of difference
from one initialization constant. This is why every serious LSTM implementation initializes the forget
bias positive.

## Key takeaways

- The LSTM's additive cell update lets memory (and gradients) survive long gaps.
- Three gates (forget/input/output) learn what to keep, write, and expose.
- The GRU achieves the same with two gates and one state — fewer parameters.
- Opening the forget gate preserves a signal indefinitely, unlike a vanilla RNN.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — LSTM vs GRU parameter counts

Every gate is one dense layer over the concatenation $[\mathbf{x}; \mathbf{h}]$, costing $d_h (d_{in} + d_h) + d_h$ parameters. The LSTM has **4** gate-shaped blocks (forget, input, candidate, output); the GRU has **3** (reset, update, candidate). Implement both counts — the ratio is exactly $4/3$.

In [ ]:
def gate_params(d_in, d_h):
    """One gate: a dense layer on [x; h] -> d_h, with bias."""
    # TODO(you): d_h * (d_in + d_h) + d_h
    return ...


def lstm_params(d_in, d_h):
    # TODO(you): 4 gates
    return ...


def gru_params(d_in, d_h):
    # TODO(you): 3 gates
    return ...

In [ ]:
# Checks — run me
assert lstm_params(1, 1) == 12, "scalar LSTM: 4 gates x (1 + 1 weights + 1 bias)"
assert lstm_params(128, 256) == 4 * (256 * 384 + 256), "4 gates, each a dense layer on [x; h]"
assert gru_params(128, 256) == 3 * (256 * 384 + 256), "GRU drops one gate"
assert abs(lstm_params(128, 256) / gru_params(128, 256) - 4 / 3) < 1e-12, "exactly 4/3 the parameters"

# Edge case: hidden size of 1 with no input (d_in=0) -- only the recurrent weight + bias remain
assert gate_params(0, 1) == 2, "d_in=0: just W_h (1x1) + bias (1) = 2 params"
assert lstm_params(0, 1) == 8 and gru_params(0, 1) == 6, "still 4x vs 3x that per-gate count"

# Edge case: d_h=0 -- a degenerate zero-width layer has no parameters at all
assert gate_params(10, 0) == 0, "no hidden units means no weights and no bias"
assert lstm_params(10, 0) == 0 and gru_params(10, 0) == 0

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gate_params(d_in, d_h):
    return d_h * (d_in + d_h) + d_h


def lstm_params(d_in, d_h):
    return 4 * gate_params(d_in, d_h)


def gru_params(d_in, d_h):
    return 3 * gate_params(d_in, d_h)
```

</details>

### Exercise 2 — The cell-state highway

The LSTM's fix for vanishing gradients is **additive** memory:

$$c_t = f_t \, c_{t-1} + i_t \, \tilde{c}_t$$

Iterate the chain given per-step gate values. The checks demonstrate the whole argument: with $f = 1, i = 0$ the memory survives 100 steps untouched; with $f < 1$ the geometric decay of the vanilla RNN comes right back; and one open input gate writes the candidate into the cell.

In [ ]:
def cell_chain(c0, forgets, inputs, candidates):
    """Run the cell-state recurrence over aligned per-step gate values."""
    c = float(c0)

    for f, i, g in zip(forgets, inputs, candidates):
        # TODO(you): the additive update c = f*c + i*g
        c = ...

    return c

In [ ]:
# Checks — run me
T = 100
assert cell_chain(5.0, [1.0] * T, [0.0] * T, [0.0] * T) == 5.0, \
    "forget = 1, input = 0: the memory survives 100 steps untouched"
assert abs(cell_chain(5.0, [0.5] * 10, [0.0] * 10, [0.0] * 10) - 5.0 * 0.5 ** 10) < 1e-12, \
    "forget < 1 decays the memory geometrically — the vanilla RNN problem returns"
assert cell_chain(0.0, [1.0, 1.0, 1.0], [0.0, 1.0, 0.0], [0.0, 7.0, 0.0]) == 7.0, \
    "one open input gate writes the candidate into the cell"

# Edge case: sequence length 1 (a single gate step) -- reduces to one additive update
assert cell_chain(2.0, [0.5], [0.5], [4.0]) == 0.5 * 2.0 + 0.5 * 4.0, \
    "a single step is just c = f*c0 + i*g, no recurrence needed"

# Edge case: zero-initialized cell state (c0=0) with a fully-closed forget gate and an
# all-zero candidate -- the cell state stays at exactly zero no matter how long it runs
assert cell_chain(0.0, [0.0] * 50, [1.0] * 50, [0.0] * 50) == 0.0, \
    "c0=0, nothing ever written in (candidate=0): the cell state never leaves zero"

# Edge case: empty sequence -- zero gate steps means the initial cell state passes through
assert cell_chain(3.5, [], [], []) == 3.5, "no gate steps at all: c_T == c_0 unchanged"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cell_chain(c0, forgets, inputs, candidates):
    c = float(c0)
    for f, i, g in zip(forgets, inputs, candidates):
        c = f * c + i * g
    return c
```

</details>

---
## 🎯 Extra practice — DML 59: a full LSTM forward pass

[DML-OpenProblem 59](https://github.com/Open-Deep-ML/DML-OpenProblem) asks for the
same four gates as the `LSTMCell` above, but as a class matching a specific
signature: `LSTM(input_size, hidden_size)` with
`forward(x, initial_hidden_state, initial_cell_state)` returning
`(outputs_per_step, final_h, final_c)`. The one wrinkle versus the cell above: the
gate inputs are the concatenation **`[h_{t-1}; x_t]`** (hidden state first, then
input) rather than `[x_t; h_{t-1}]` — get the stacking order right or the gates
silently see the wrong slice of the weight matrix.

In [ ]:
class LSTM:
    """DML 59 — an LSTM exposing forward() with an explicit initial (h, c)."""

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.Wf = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wi = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wc = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wo = np.random.randn(hidden_size, input_size + hidden_size)
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bc = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def forward(self, x, initial_hidden_state, initial_cell_state):
        """Process a sequence; return (hidden state at every step, final h, final c)."""
        h = initial_hidden_state
        c = initial_cell_state
        outputs = []
        for t in range(len(x)):
            xt = x[t].reshape(-1, 1)
            concat = np.vstack((h, xt))          # [h_{t-1}; x_t] -- order matters!

            # TODO(you): forget gate, input gate, candidate cell state, output gate
            ft = ...
            it = ...
            c_tilde = ...
            ot = ...

            c = ft * c + it * c_tilde             # additive cell update
            h = ot * np.tanh(c)
            outputs.append(h)
        return np.array(outputs), h, c

In [ ]:
# Checks — run me (both cases match DML 59's own tests.json, with explicit weights
# for reproducibility -- same idea as the hand-worked timestep in section 2 above)
lstm = LSTM(input_size=1, hidden_size=1)
lstm.Wf = np.array([[0.5, 0.5]]); lstm.Wi = np.array([[0.5, 0.5]])
lstm.Wc = np.array([[0.3, 0.3]]); lstm.Wo = np.array([[0.5, 0.5]])
lstm.bf = np.array([[0.1]]); lstm.bi = np.array([[0.1]])
lstm.bc = np.array([[0.1]]); lstm.bo = np.array([[0.1]])
_, final_h, _ = lstm.forward(np.array([[1.0], [2.0], [3.0]]), np.zeros((1, 1)), np.zeros((1, 1)))
assert np.allclose(final_h, [[0.73698596]], atol=1e-6), "DML 59 test case 1"

lstm2 = LSTM(input_size=2, hidden_size=2)
lstm2.Wf = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm2.Wi = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm2.Wc = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm2.Wo = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]])
lstm2.bf = np.array([[0.1], [0.2]]); lstm2.bi = np.array([[0.1], [0.2]])
lstm2.bc = np.array([[0.1], [0.2]]); lstm2.bo = np.array([[0.1], [0.2]])
_, final_h2, _ = lstm2.forward(np.array([[0.1, 0.2], [0.3, 0.4]]), np.zeros((2, 1)), np.zeros((2, 1)))
assert np.allclose(final_h2, [[0.16613133], [0.40299449]], atol=1e-6), "DML 59 test case 2"

# Edge case: sequence length 1 -- exactly one gate/cell/hidden update from the initial state
_, final_h3, final_c3 = lstm.forward(np.array([[1.0]]), np.zeros((1, 1)), np.zeros((1, 1)))
assert np.allclose(final_h3, [[0.15528746]], atol=1e-6), "one step in: matches a hand-checked value"
assert np.allclose(final_c3, [[0.24531644]], atol=1e-6)

# Edge case: zero-initialized hidden AND cell state with an all-zero input sequence --
# the bias terms alone still move the gates and cell state away from zero
_, final_h4, final_c4 = lstm.forward(np.array([[0.0], [0.0], [0.0]]), np.zeros((1, 1)), np.zeros((1, 1)))
assert not np.allclose(final_h4, 0.0), "nonzero gate biases still write into the cell state"
assert np.allclose(final_h4, [[0.05539166]], atol=1e-6) and np.allclose(final_c4, [[0.10478854]], atol=1e-6)

# Edge case: fully zero weights AND zero biases -- sigmoid(0)=0.5, tanh(0)=0, so the cell
# state and hidden state stay exactly zero forever, regardless of how many steps run
lstm_zero = LSTM(input_size=1, hidden_size=1)
for name in ["Wf", "Wi", "Wc", "Wo"]:
    setattr(lstm_zero, name, np.zeros((1, 2)))
_, final_h5, final_c5 = lstm_zero.forward(np.array([[0.0], [0.0], [0.0], [0.0]]), np.zeros((1, 1)), np.zeros((1, 1)))
assert np.allclose(final_h5, 0.0) and np.allclose(final_c5, 0.0), \
    "all-zero weights and biases: the cell state can never leave zero"

print("✅ Extra-practice DML 59 (LSTM forward pass) passed")

<details>
<summary>💡 Show solution</summary>

```python
class LSTM:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.Wf = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wi = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wc = np.random.randn(hidden_size, input_size + hidden_size)
        self.Wo = np.random.randn(hidden_size, input_size + hidden_size)
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bc = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def forward(self, x, initial_hidden_state, initial_cell_state):
        h = initial_hidden_state
        c = initial_cell_state
        outputs = []
        for t in range(len(x)):
            xt = x[t].reshape(-1, 1)
            concat = np.vstack((h, xt))
            ft = self.sigmoid(self.Wf @ concat + self.bf)
            it = self.sigmoid(self.Wi @ concat + self.bi)
            c_tilde = np.tanh(self.Wc @ concat + self.bc)
            ot = self.sigmoid(self.Wo @ concat + self.bo)
            c = ft * c + it * c_tilde
            h = ot * np.tanh(c)
            outputs.append(h)
        return np.array(outputs), h, c
```

</details>